In [ ]:
# Install necessary packages
!pip install opencv-python-headless matplotlib

# Clone the SAM repository from Meta
!git clone https://github.com/facebookresearch/segment-anything.git

# Download the SAM ViT-H checkpoint
!wget -O sam_vit_h_4b8939.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

Cloning into 'segment-anything'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 304 (delta 2), reused 1 (delta 1), pack-reused 299 (from 2)
Receiving objects: 100% (304/304), 18.31 MiB | 16.19 MiB/s, done.
Resolving deltas: 100% (159/159), done.
--2025-02-07 19:37:03--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.227.219.59, 13.227.219.70, 13.227.219.10, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.227.219.59|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2564550879 (2.4G) [binary/octet-stream]
Saving to: ‘sam_vit_h_4b8939.pth’

sam_vit_h_4b8939.pt 100%[===================>]   2.39G   169MB/s    in 18s     

2025-02-07 19:37:22 (132 MB/s) - ‘sam_vit_h_4b8939.pth’ saved [2564550879/2564550879]



In [ ]:
import sys
sys.path.append('segment-anything')

import cv2
import numpy as np
import torch
import os
from segment_anything import SamPredictor, sam_model_registry, SamAutomaticMaskGenerator

# ----- CONFIGURATION -----
device = "cuda" if torch.cuda.is_available() else "cpu"
sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"

# ----- LOAD THE MODEL -----
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)
mask_generator = SamAutomaticMaskGenerator(sam)

/content/segment-anything/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


Create a Folder "images" and paste all the images in that.

In [ ]:

# ----- SETUP DIRECTORIES -----
input_dir = "/content/images"
output_dir = "/content/output"
os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# ----- PROCESS IMAGES -----
image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff')

for filename in os.listdir(input_dir):
    if not filename.lower().endswith(image_extensions):
        continue

    image_path = os.path.join(input_dir, filename)
    print(f"\nProcessing: {filename}")

    # Load image
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        print(f"Error loading {filename}, skipping...")
        continue
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    # Create output subfolder
    base_name = os.path.splitext(filename)[0]
    image_output_dir = os.path.join(output_dir, base_name)
    os.makedirs(image_output_dir, exist_ok=True)

    # ----- MANUAL BBOX PROCESSING -----
    try:
        predictor.set_image(image_rgb)
        bbox = np.array([100, 200, 400, 800])  # Adjust this per your needs
        masks, _, _ = predictor.predict(box=bbox, multimask_output=True)
        if len(masks) > 0:
            mask = masks[0].astype(np.uint8) * 255
            # Save mask (unchanged)
            cv2.imwrite(os.path.join(image_output_dir, "manual_mask.png"), mask)

            # Create white background and apply mask
            white_bg = np.ones_like(image_bgr) * 255
            cloth = np.where(mask[..., None], image_bgr, white_bg)
            cv2.imwrite(os.path.join(image_output_dir, "manual_cloth.png"), cloth)
    except Exception as e:
        print(f"Manual processing failed: {str(e)}")

    # ----- AUTOMATIC MASK GENERATION -----
    try:
        all_masks = mask_generator.generate(image_rgb)
        for idx, mask_data in enumerate(all_masks):
            mask = mask_data['segmentation'].astype(np.uint8) * 255
            # Save mask (unchanged)
            cv2.imwrite(os.path.join(image_output_dir, f"auto_mask_{idx}.png"), mask)

            # Create white background and apply mask
            white_bg = np.ones_like(image_bgr) * 255
            cloth = np.where(mask[..., None], image_bgr, white_bg)
            cv2.imwrite(os.path.join(image_output_dir, f"auto_cloth_{idx}.png"), cloth)

        print(f"Generated {len(all_masks)} automatic masks")
    except Exception as e:
        print(f"Automatic processing failed: {str(e)}")


Processing: 00U1SDY24V87_1.jpg
Generated 17 automatic masks

Processing: 23_M_MenKurtaPajamaTailored_EMTSC23-104_1.jpg
Generated 18 automatic masks

Processing: 0MST2P24CPL6_1.jpg
Generated 17 automatic masks


In [ ]:
!zip -r /content/output2.zip /content/output

  adding: content/output/ (stored 0%)
  adding: content/output/23_M_MenKurtaPajamaTailored_EMTSC23-104_1/ (stored 0%)
  adding: content/output/23_M_MenKurtaPajamaTailored_EMTSC23-104_1/auto_cloth_2.png (deflated 34%)
  adding: content/output/23_M_MenKurtaPajamaTailored_EMTSC23-104_1/auto_cloth_6.png (deflated 37%)
  adding: content/output/23_M_MenKurtaPajamaTailored_EMTSC23-104_1/auto_cloth_13.png (deflated 43%)
  adding: content/output/23_M_MenKurtaPajamaTailored_EMTSC23-104_1/auto_cloth_17.png (deflated 26%)
  adding: content/output/23_M_MenKurtaPajamaTailored_EMTSC23-104_1/auto_cloth_8.png (deflated 29%)
  adding: content/output/23_M_MenKurtaPajamaTailored_EMTSC23-104_1/auto_mask_7.png (deflated 84%)
  adding: content/output/23_M_MenKurtaPajamaTailored_EMTSC23-104_1/auto_mask_10.png (deflated 82%)
  adding: content/output/23_M_MenKurtaPajamaTailored_EMTSC23-104_1/auto_cloth_11.png (deflated 33%)
  adding: content/output/23_M_MenKurtaPajamaTailored_EMTSC23-104_1/auto_mask_3.png (defl